# Nuova-Norma: vettori degli articoli

Calcola i vettori degli articoli della Raccolta sistematica, una sola volta, su GPU. I testi sono **pubblici** (Fedlex):
qui non arriva nessuna domanda di utenti. Le domande si codificano poi sul PC, in locale, con lo stesso modello.

1. Sul PC: `py -m nn esporta-vettori` crea `data/vettori/da-calcolare.jsonl`.
2. Qui: *Runtime > Cambia tipo di runtime > GPU* (A100 o L4), poi esegui le celle in ordine.
3. Sul PC: `py -m nn importa-vettori vettori.npz`.

Il modello deve essere lo stesso del PC (`NN_VETTORI_MODELLO`, predefinito `BAAI/bge-m3`, 1024 dimensioni).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip -q install -U sentence-transformers

## Il file da calcolare
Due modi: caricarlo dal computer (fino a qualche centinaio di MB va bene) oppure da Google Drive.

In [ ]:
MODELLO = "BAAI/bge-m3"
DIM = 1024
USA_DRIVE = False   # True: legge e scrive in MyDrive/nuova-norma/

import os
if USA_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    CARTELLA = "/content/drive/MyDrive/nuova-norma"
    SORGENTE = f"{CARTELLA}/da-calcolare.jsonl"
else:
    from google.colab import files
    CARTELLA = "/content"
    caricati = files.upload()
    SORGENTE = f"{CARTELLA}/" + next(iter(caricati))
print(SORGENTE, os.path.getsize(SORGENTE) // 1_000_000, "MB")

In [ ]:
import json
r, e, h, t = [], [], [], []
with open(SORGENTE, encoding="utf-8") as f:
    for riga in f:
        d = json.loads(riga)
        r.append(d["r"]); e.append(d["e"]); h.append(d["h"]); t.append(d["t"])
print(len(t), "articoli")

## Calcolo

In [ ]:
import numpy as np, torch, time
from sentence_transformers import SentenceTransformer

modello = SentenceTransformer(MODELLO, device="cuda", model_kwargs={"torch_dtype": torch.float16})
modello.max_seq_length = 2048   # gli articoli sono tagliati a 6000 caratteri dall'esportazione

ordine = np.argsort([len(x) for x in t])          # testi di lunghezza simile insieme: molto piu veloce
t0 = time.time()
v = modello.encode([t[i] for i in ordine], batch_size=64, normalize_embeddings=True,
                   show_progress_bar=True, convert_to_numpy=True)
vettori = np.empty_like(v)
vettori[ordine] = v
assert vettori.shape[1] == DIM, vettori.shape
print(f"{len(t)} articoli in {time.time() - t0:.0f} s")

In [ ]:
USCITA = f"{CARTELLA}/vettori.npz"
np.savez(USCITA, r=np.asarray(r, dtype="int64"), e=np.asarray(e, dtype="U"), h=np.asarray(h, dtype="U"),
         v=vettori.astype("float16"))
print(USCITA, os.path.getsize(USCITA) // 1_000_000, "MB")
if not USA_DRIVE:
    files.download(USCITA)

## Controllo veloce
Una domanda di prova: gli articoli piu vicini devono avere senso (stesso modello, stessa normalizzazione del PC).

In [ ]:
prova = "Il datore di lavoro puo licenziarmi mentre sono malato?"
q = modello.encode([prova], normalize_embeddings=True)[0]
for i in np.argsort(-(vettori.astype("float32") @ q))[:5]:
    print(round(float(vettori[i].astype("float32") @ q), 3), t[i][:120].replace("\n", " | "))